# 🏈 Live Draft Assistant (runs in your browser — no install)

A real-time draft war room for your ESPN fantasy football league. It watches
your draft as picks happen and tells you the best pick for **your** team,
blending player value, your roster needs, and positional scarcity.

**Nothing to install.** This runs on Google's free servers, not your computer.

## How to use it
1. Run **Cell 1** (setup) — click it and press the ▶ button, or `Shift+Enter`.
2. Fill in your details in **Cell 2**, then run it.
3. Run **Cell 3** and leave the tab open during your draft. It refreshes itself.

To stop it, press the ⏹ (stop) button next to Cell 3.

> **Privacy:** your ESPN cookies stay in your own Colab session. Don't share
> this notebook with the cookies filled in, and use *Runtime → Disconnect and
> delete runtime* when you're done.

## Cell 1 — Setup (run once)

In [ ]:
!pip -q install espn_api
!rm -rf /content/espn-api
!git clone -q -b claude/repo-draft-help-h3i9nw https://github.com/imlevelhead/espn-api /content/espn-api
import sys
sys.path.insert(0, '/content/espn-api/examples')
import draft_assistant as da
print('Setup complete. Now fill in Cell 2.')

## Cell 2 — Your league details (edit, then run)

**Getting your cookies (private league):** log in at fantasy.espn.com in
Chrome/Edge → press `F12` → **Application** tab → **Cookies** →
`https://fantasy.espn.com`. Copy the *Value* of `espn_s2` and `SWID`
(keep the `{ }` braces on SWID).

In [ ]:
# ---- EDIT THESE ----
da.LEAGUE_ID = 123456          # your league id (from the league URL: leagueId=...)
da.YEAR      = 2026            # draft season

da.ESPN_S2   = 'PASTE_ESPN_S2_HERE'   # private league only
da.SWID      = '{PASTE-SWID-HERE}'    # private league only (keep the braces)

# Identify your team: set ONE of these (leave the other as None).
da.MY_TEAM_NAME = None        # e.g. 'Griddy Boys'  (exact name, case-insensitive)
da.MY_TEAM_ID   = None        # e.g. 3             (run Cell 3 once to see team ids)

da.REFRESH_SECONDS = 8        # how often to re-check the draft
print('Details saved. Now run Cell 3.')

## Cell 3 — Run the war room (start before your draft, leave it running)

In [ ]:
import time
from IPython.display import clear_output

print('Connecting to ESPN...')
league = da.League(league_id=da.LEAGUE_ID, year=da.YEAR, espn_s2=da.ESPN_S2, swid=da.SWID)
print(f'Connected: {len(league.teams)} teams.')

my_team = da.resolve_my_team(league)
if my_team is None:
    print('\nCould not match your team. Here are the teams in your league:')
    for t in league.teams:
        print(f'   id={t.team_id}  {t.team_name}')
    print('\nSet da.MY_TEAM_ID (or da.MY_TEAM_NAME) in Cell 2, re-run it, then re-run this cell.')
else:
    board = da.build_board(league)

    def render_text(league, my_team, board, picks):
        lines = []
        drafted_ids = {p.playerId for p in picks}
        starters = da.get_starters(league)
        n_teams = len(league.teams)
        id_to_board = {p['id']: p for p in board}
        my_players = []
        for pick in picks:
            if pick.team is not None and my_team is not None and pick.team.team_id == my_team.team_id:
                bp = id_to_board.get(pick.playerId)
                if bp:
                    my_players.append(bp)
        slot = da.my_draft_slot(league, my_team, picks)
        until = da.picks_until_my_turn(slot, n_teams, len(picks))
        recs, best_by_pos, needs = da.recommend(board, drafted_ids, my_players, starters)
        lines.append('=' * 68)
        lines.append(f' DRAFT WAR ROOM — {league.league_id} / {league.year}   picks made: {len(picks)}')
        turn = ("YOU'RE ON THE CLOCK ⏰" if until == 0 else f'{until} pick(s) until your turn' if until is not None else 'waiting for your first pick...')
        lines.append(f' You: {my_team.team_name}  (slot {slot})   {turn}')
        lines.append('=' * 68)
        need_str = ' '.join(f'{k}:{v}' for k, v in needs.items() if v > 0) or 'none — all starters filled'
        lines.append(f'\n YOUR ROSTER ({len(my_players)}):')
        for p in my_players:
            lines.append('   ' + da.fmt_player(p))
        if not my_players:
            lines.append('   (empty)')
        lines.append(f'\n STILL NEED (starters): {need_str}')
        lines.append('\n RECOMMENDED PICKS (value + your needs + scarcity):')
        for i, (score, scarce, needed, p) in enumerate(recs, 1):
            tags = []
            if needed:
                tags.append('fills need')
            if scarce:
                tags.append('tier thinning')
            tag = f"  <- {', '.join(tags)}" if tags else ''
            lines.append(f'  {i}. ' + da.fmt_player(p) + tag)
        if not recs:
            lines.append('   (no players available — is the draft over?)')
        lines.append('\n BEST AVAILABLE BY POSITION:')
        for pos in da.SCORING_POSITIONS:
            pool = best_by_pos.get(pos)
            if pool:
                lines.append(f'   {pos:5}: ' + da.fmt_player(pool[0]))
        lines.append(f'\n (auto-refreshing every {da.REFRESH_SECONDS}s — press the stop button to quit)')
        return '\n'.join(lines)

    try:
        while True:
            picks = da.refresh_picks(league)
            clear_output(wait=True)
            print(render_text(league, my_team, board, picks))
            time.sleep(da.REFRESH_SECONDS)
    except KeyboardInterrupt:
        print('\nStopped. Good luck — go win your league. 🏆')